In [69]:
import pandas as pd
import numpy as np
import os
import re

# 1. CONCATENATE TYPES OF OBJECTS

(Space Stations, Sattelites, Debris, Analyst)

In [70]:
# READ FILES
space_station_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Space_Station/space_station.csv")
sattelites_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Sattelites/celestrack_active_sattelites.csv")
analyst_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Analyst/analyst_sattelites.csv")
cosmos_1408_deb_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Debris/cosmos_1408.csv")
cosmos_2251_deb_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Debris/cosmos_2251.csv")
fengyun_1c_deb_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Debris/fengyun_1c.csv")
iridium_33_deb_df = pd.read_csv("DATASETS_SATTELITES/celestrack/Debris/iridium_33.csv")

# DEFINE LISTS TO HELP WITH DATASET CREATION
df_names = ["space_station_df", "sattelites_df", "analyst_df", "cosmos_1408_deb_df", "cosmos_2251_deb_df", "fengyun_1c_deb_df", "iridium_33_deb_df"]
df_list = [space_station_df, sattelites_df, analyst_df, cosmos_1408_deb_df, cosmos_2251_deb_df, fengyun_1c_deb_df, iridium_33_deb_df]
df_types = ["Space Station", "Sattelite", "Analyst", "Debris", "Debris", "Debris", "Debris"]

# COUNT AND ADD NEW COLUMN (OBJECT TYPE)
total_objects = 0
print("DATASETS SIZES:")
for df, name, type in zip(df_list, df_names, df_types):
    cur_objects = len(df.index)
    total_objects += cur_objects
    df.insert(2, 'OBJECT_TYPE', type)   # third column
    print(f"{name:20}: {cur_objects}")

print("\nTOTAL OBJECTS:", total_objects)

DATASETS SIZES:
space_station_df    : 34
sattelites_df       : 14408
analyst_df          : 444
cosmos_1408_deb_df  : 4
cosmos_2251_deb_df  : 581
fengyun_1c_deb_df   : 1844
iridium_33_deb_df   : 109

TOTAL OBJECTS: 17424


In [71]:
# CONCATENATE DATA
total_df = pd.concat([space_station_df, sattelites_df, analyst_df, cosmos_1408_deb_df, cosmos_2251_deb_df, fengyun_1c_deb_df, iridium_33_deb_df])

### Check duplicates

In [ ]:
duplicados = total_df[total_df["NORAD_CAT_ID"].duplicated()]
len(duplicados)

30

In [91]:
# 1. Filtra mantendo TODAS as instâncias dos IDs duplicados
# 2. Ordena por ID para que os gémeos fiquem colados um ao outro
duplicados_completos = total_df[total_df.duplicated(subset="NORAD_CAT_ID", keep=False)].sort_values("NORAD_CAT_ID")

# 3. Imprime sem truncar as linhas (para veres a lista toda)
import pandas as pd
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(duplicados_completos[["NORAD_CAT_ID", "OBJECT_NAME", "OBJECT_TYPE"]])

       NORAD_CAT_ID      OBJECT_NAME    OBJECT_TYPE
0             25544      ISS (ZARYA)  Space Station
60            25544      ISS (ZARYA)      Sattelite
2267          48274     CSS (TIANHE)      Sattelite
2             48274     CSS (TIANHE)  Space Station
3             49044      ISS (NAUKA)  Space Station
2603          49044      ISS (NAUKA)      Sattelite
3943          53239    CSS (WENTIAN)      Sattelite
5             53239    CSS (WENTIAN)  Space Station
6             54216   CSS (MENGTIAN)  Space Station
4610          54216   CSS (MENGTIAN)      Sattelite
11629         64751   PROGRESS-MS 31      Sattelite
7             64751   PROGRESS-MS 31  Space Station
8             64786       TIANZHOU-9  Space Station
11660         64786       TIANZHOU-9      Sattelite
12364         65586   PROGRESS-MS 32      Sattelite
9             65586   PROGRESS-MS 32  Space Station
10            65616     CYGNUS NG-23  Space Station
12391         65616     CYGNUS NG-23      Sattelite
12488       

We notice the duplicates are characterizes as both sattelite and space station. 
For the porpuse of this project, we decided to characterize them as Space Station, since that object type is more "special".

In [92]:
# 1. Identificamos quais são os IDs que aparecem mais de uma vez
ids_duplicados = total_df[total_df.duplicated('NORAD_CAT_ID', keep=False)]['NORAD_CAT_ID'].unique()

# 2. Mantemos apenas as linhas que:
# (NÃO têm um ID duplicado) OU (têm ID duplicado mas o tipo NÃO é 'Sattelite')
total_df = total_df[~(total_df['NORAD_CAT_ID'].isin(ids_duplicados) & (total_df['OBJECT_TYPE'] == 'Sattelite'))]

### Check duplicates again

In [93]:
len(total_df[total_df.duplicated('NORAD_CAT_ID')])

0

# 2. Check files

In [72]:
# FOLDERS
base_folder = "DATASETS_SATTELITES/celestrack"

weather_folder = f"{base_folder}/Weather"
communication_folder = f"{base_folder}/Communication"
brightest_folder = f"{base_folder}/Brightest"
earth_res_folder = f"{base_folder}/Earth_Resources"
misc_folder = f"{base_folder}/Miscellaneous"
nav_folder = f"{base_folder}/Navigation"
scientific_folder = f"{base_folder}/Scientific"

folder_list = [weather_folder, communication_folder, brightest_folder, earth_res_folder, misc_folder, nav_folder, scientific_folder]
filter_list = ["WEATHER", "COMMUNICATION", "BRIGHTEST", "EARTH_RESOURCES", "MISCELLANEOUS", "NAVIGATION", "SCIENTIFIC"]


Check if there are values on the datasets, not present in their respective filter all.csv file

In [94]:
for folder, category in zip(folder_list, filter_list):
    if os.path.exists(folder):
        files = os.listdir(folder)
        
        # 1. Localizar e carregar o ficheiro "ALL" primeiro
        all_file = [f for f in files if "all" in f.lower() and f.endswith(".csv")]
        
        if all_file:
            all_path = os.path.join(folder, all_file[0])
            # Verificamos se o 'all' não está vazio para evitar o erro que tiveste
            if os.path.getsize(all_path) > 0:
                df_all = pd.read_csv(all_path)
                ids_all = set(df_all['NORAD_CAT_ID'].unique())
            else:
                print(f"❌ Erro: O ficheiro mestre {all_file[0]} está vazio.")
                continue
        else:
            print(f"⚠️ Aviso: Não foi encontrado ficheiro 'ALL' na pasta {folder}.")
            print("-"*30)
            continue

        # 2. Comparar cada ficheiro específico contra o "ALL"
        for filename in files:
            if filename.endswith(".csv") and "all" not in filename.lower():
                file_path = os.path.join(folder, filename)
                
                if os.path.getsize(file_path) > 0:
                    try:
                        this_df = pd.read_csv(file_path)
                        ids_specific = set(this_df['NORAD_CAT_ID'].unique())
                        
                        # Diferença: IDs que estão no específico mas NÃO no 'all'
                        orphans = ids_specific - ids_all
                        
                        if orphans:
                            print(f"❗ Inconsistência em {category} -> {filename}:")
                            print(f"   {len(orphans)} IDs presentes aqui não aparecem no ficheiro 'ALL'.")
                            print("-"*30)
                            # print(f"   IDs: {orphans}") # Opcional: ver os IDs reais
                    except Exception as e:
                        print(f"❌ Erro ao ler {filename}: {e}")

❗ Inconsistência em WEATHER -> goes.csv:
   13 IDs presentes aqui não aparecem no ficheiro 'ALL'.
------------------------------
❗ Inconsistência em WEATHER -> noah.csv:
   20 IDs presentes aqui não aparecem no ficheiro 'ALL'.
------------------------------
⚠️ Aviso: Não foi encontrado ficheiro 'ALL' na pasta DATASETS_SATTELITES/celestrack/Communication.
------------------------------
❗ Inconsistência em EARTH_RESOURCES -> argos.csv:
   29 IDs presentes aqui não aparecem no ficheiro 'ALL'.
------------------------------
❗ Inconsistência em EARTH_RESOURCES -> disaster_monitoring.csv:
   5 IDs presentes aqui não aparecem no ficheiro 'ALL'.
------------------------------
❗ Inconsistência em EARTH_RESOURCES -> planet.csv:
   61 IDs presentes aqui não aparecem no ficheiro 'ALL'.
------------------------------
❗ Inconsistência em EARTH_RESOURCES -> sarsat.csv:
   84 IDs presentes aqui não aparecem no ficheiro 'ALL'.
------------------------------
❗ Inconsistência em EARTH_RESOURCES -> spire.

Check if in each folder, there are objects simultaneous in differente filter types

For example: sattelite being both education and engineering

In [96]:
for folder, category in zip(folder_list, filter_list):
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.endswith(".csv") and "all" not in f.lower()]
        
        # Dicionário para mapear: {ID_do_satelite: [lista_de_ficheiros_onde_aparece]}
        seen_ids = {}

        print(f"--- Verificando duplicados internos em: {category} ---")
        
        for filename in files:
            file_path = os.path.join(folder, filename)
            
            if os.path.getsize(file_path) > 0:
                try:
                    this_df = pd.read_csv(file_path)
                    ids_in_file = this_df['OBJECT_ID'].unique()
                    
                    for sat_id in ids_in_file:
                        if sat_id in seen_ids:
                            seen_ids[sat_id].append(filename)
                        else:
                            seen_ids[sat_id] = [filename]
                            
                except Exception as e:
                    print(f"❌ Erro ao ler {filename}: {e}")

        # Após ler todos os ficheiros da pasta, reportar quem aparece em mais de um
        duplicates_found = False
        for sat_id, filenames in seen_ids.items():
            if len(filenames) > 1:
                duplicates_found = True
                print(f"🔍 Satélite {sat_id} repetido em: {', '.join(filenames)}")
        
        if not duplicates_found:
            print("✅ Nenhum satélite repetido entre os ficheiros específicos.")
        
        print("-" * 40)

--- Verificando duplicados internos em: WEATHER ---
✅ Nenhum satélite repetido entre os ficheiros específicos.
----------------------------------------
--- Verificando duplicados internos em: COMMUNICATION ---
✅ Nenhum satélite repetido entre os ficheiros específicos.
----------------------------------------
--- Verificando duplicados internos em: BRIGHTEST ---
✅ Nenhum satélite repetido entre os ficheiros específicos.
----------------------------------------
--- Verificando duplicados internos em: EARTH_RESOURCES ---
🔍 Satélite 2012-049A repetido em: argos.csv, sarsat.csv
----------------------------------------
--- Verificando duplicados internos em: MISCELLANEOUS ---
✅ Nenhum satélite repetido entre os ficheiros específicos.
----------------------------------------
--- Verificando duplicados internos em: NAVIGATION ---
🔍 Satélite 2018-085A repetido em: augmentation_system.csv, beidou.csv
🔍 Satélite 2020-017A repetido em: augmentation_system.csv, beidou.csv
🔍 Satélite 2020-040A repet

# 3. ADD METADATA

Weather, Communication, Brightest, Earth Resources, Miscellaneaour, Navigation, Scientific

In [ ]:

for folder, type in zip(folder_list, filter_list):
    # VERIFY IF THE FOLDER EXISTES
    if os.path.exists(folder):
        # List all files inside folder
        files = os.listdir(folder)

        # set vlues to "False"
        total_df[type] = "False"

        if type != "COMMUNICATION":
            for filename in files:
                # Only files (ignores folders)
                if filename.endswith(".csv"):
                    # Check name
                    if "all" in filename.lower():
                        # TODO: O que fazer quando o ficheiro tem "all" no nome
                        pass
                    else:
                        # TODO: O que fazer para os ficheiros específicos (sem "all")
                        # 1. Construir o caminho completo e ler o CSV específico
                        file_path = os.path.join(folder, filename)
                        this_df = pd.read_csv(file_path)
                        
                        # 2. Extrair os IDs (garantindo que o nome da coluna coincide, ex: 'NORAD_CAT_ID')
                        # ... (o teu código anterior)
                        ids_on_csv = this_df['NORAD_CAT_ID'].unique()
                        
                        # --- NOVA VERIFICAÇÃO ---
                        # Criamos conjuntos (sets) para comparar as listas de IDs
                        set_total = set(total_df['NORAD_CAT_ID'])
                        set_csv = set(ids_on_csv)

                        # IDs que estão no CSV mas NÃO estão no total_df
                        missing_ids = set_csv - set_total

                        if len(missing_ids) > 0:
                            print(f"⚠️ Aviso em {filename}: {len(missing_ids)} IDs não encontrados no total_df.")
                            # Opcional: print(f"IDs em falta: {missing_ids}")
                        # -------------------------

                        new_name_type = filename.replace('.csv', '')
                        
                        # Continua com a atualização...
                        total_df.loc[total_df['NORAD_CAT_ID'].isin(ids_on_csv), type] = new_name_type
        else:
            # --- LÓGICA ESPECÍFICA PARA COMMUNICATION ---
            subfolders = ["Active_Geosynchronous", "Constellations"]
            
            # 1. Criar as duas colunas fixas no total_df
            total_df["active geosynchronous"] = "False"
            total_df["constellations"] = "False"

            for sub in subfolders:
                sub_path = os.path.join(folder, sub)
                
                if os.path.exists(sub_path):
                    # Definimos qual a coluna a atualizar baseada na subpasta
                    target_column = sub.lower() 
                    
                    comm_files = os.listdir(sub_path)
                    for filename in comm_files:
                        if filename.endswith(".csv") and "all" not in filename.lower():
                            file_path = os.path.join(sub_path, filename)
                            this_df = pd.read_csv(file_path)
                            ids_on_csv = this_df['NORAD_CAT_ID'].unique()

                            # O nome do "tipo" é o nome do ficheiro (ex: 'starlink')
                            new_name_type = filename.replace('.csv', '')
                            
                            # 2. Atualizar a coluna correspondente (active geosynchronous ou constellations)
                            total_df.loc[total_df['NORAD_CAT_ID'].isin(ids_on_csv), target_column] = new_name_type
                else:
                    print(f"WARNING: Subfolder not found - {sub_path}")
    else:
        print(f"WARNING: Folder not founded - {folder}")

⚠️ Aviso em goes.csv: 13 IDs não encontrados no total_df.
⚠️ Aviso em noah.csv: 20 IDs não encontrados no total_df.
⚠️ Aviso em nnss.csv: 18 IDs não encontrados no total_df.
⚠️ Aviso em russian_leo_navigation.csv: 6 IDs não encontrados no total_df.


In [60]:
teste_satelite = total_df.loc[total_df["NORAD_CAT_ID"] == 6920]
print(teste_satelite)

Empty DataFrame
Columns: [OBJECT_NAME, OBJECT_ID, OBJECT_TYPE, EPOCH, MEAN_MOTION, ECCENTRICITY, INCLINATION, RA_OF_ASC_NODE, ARG_OF_PERICENTER, MEAN_ANOMALY, EPHEMERIS_TYPE, CLASSIFICATION_TYPE, NORAD_CAT_ID, ELEMENT_SET_NO, REV_AT_EPOCH, BSTAR, MEAN_MOTION_DOT, MEAN_MOTION_DDOT, WEATHER, BRIGHTEST, EARTH_RESOURCES, NAVIGATION, SCIENTIFIC, MISCELLANEOUS, COMMUNICATION]
Index: []

[0 rows x 25 columns]


In [68]:
# SAVE FILE
total_df.to_csv("DATASETS_SATTELITES/celestrack/dataset_concat.csv", index=False, )